## Minimal working example of the uproot workflow

This notebook is an example mini-analysis that loads generator predictions into memory, manipulates the MC dataset, and produces a plot of final state particle kinematics. We offload some responsibility to "library" code living in the "lehad" package (this is code that we want to share among notebooks/analyses).

The workflow for an uproot analysis is:
1: Read an entire dataset into memory given a preselection,
2: Transform the dataset into kinematic quantities by event for quantities you're interested in,
3: Supply the transformed dataset to matplotlib with weights to compute a differential cross section as a function of your kinematic quantities.

Install requirements commonly used for generator studies and import our custom libary code as a "package."

In [ ]:
! pip install -r ../requirements.txt
import sys
sys.path.append("..")

Perform imports for this notebook.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import uproot

import lehad

In [ ]:
# The cuts are the selection: Don't include any events failing these criteria.
# Under the hood, I think this is performing the boolean comparison operations numpy style which is why the syntax is a little restrictive.
# (Yes, this needs to use bitwise & not logical and.)
# This would need to be a nice place to apply a "preselection" where we throw out events
#   we definitely won't ever care about.
CUTS = """
(Enu < 4.5) & \
(flagCC) \
"""

Load some generator predictions from disk. This is a slow step, so only do this once.

In [ ]:
print("Querying filepaths...")
filepaths = lehad.get_generator_filepaths(lehad.GeneratorType.G18,lehad.BeamType.FHC,lehad.NeutrinoType.Numu)
print(f"Got: {filepaths}")
# For this minimal example, just use the first file.
input_path = filepaths[0]
n_files = 1 # In general, we will use the number of input files to normalize the cross section. In this case, it's only 1.

print(f'Opening input {input_path}...')
# Uproot works with an input file handle.
with uproot.open(input_path) as infile:
  # The "tree" is an object inside the input file - it's a columnar dataset.
  tree = infile['generator_data']
  # We can investigate the tree before reading objects like so:
  print(tree.typenames())
  
  # Access the tree by asking for one (or multiple) of its branches.
  # This is a dict { branch name -> branch }
  #   where "branch" is some structure indexed by event. It can be a list of scalars, a list of four-vectors, etc.
  print('Registering branches to enable and loading from disk...')
  enabled_branches = tree.arrays(
    ['Enu','n_FS_Protons','n_FS_PiPs','n_FS_PiMs','n_FS_Pi0s','FS_Protons','EventWeight','GenScaleFactor'],
  CUTS)
    
# If we wanted to run over multiple files, we would need to use n_files > 1 to get the cross section normalization correct
#   and concatenate the "dataframes" in memory.

Perform analysis on the generator predictions! As an example, we've computed the sum of proton kinetic energies for all events passing our preselection.

In [ ]:
  # Compute the weights per event with this prescription.
  print('Computing weights...')
  weights = enabled_branches['EventWeight']*enabled_branches['GenScaleFactor']/n_files
  
  print('Printing...')
  print(enabled_branches)
  print('Enu:')
  print(enabled_branches['Enu'])
  print('n_FS_Protons:') # Expect a list of integers.
  print(enabled_branches['n_FS_Protons'])
  print('FS_Protons:') # Expect a list of lists of four-vectors.
  # First dereference by event index then dereference by FS proton index for this event.
  print(enabled_branches['FS_Protons'])
  print('FS_Protons with no pions:')
  # It's also legal to apply cuts at this stage like so: (elements of the logical expression do need to be in parantheses.)
  n_FS_PiPs = enabled_branches['n_FS_PiPs']
  n_FS_PiMs = enabled_branches['n_FS_PiMs']
  n_FS_Pi0s = enabled_branches['n_FS_Pi0s']
  print(enabled_branches['FS_Protons'][(n_FS_PiPs == 0) & (n_FS_PiMs == 0) & (n_FS_Pi0s == 0)])
  print('EventWeight:') # Expect a list of scalars.
  print(enabled_branches['EventWeight'])
  print('GenScaleFactor:') # Expect a list of scalars.
  print(enabled_branches['GenScaleFactor'])
  
  # We'll often want to interpret functions of the raw four-vectors.
  # Use python columnar manipulation to get proton energies.
  print('Proton energies:') # (The first element of a four-vector is its energy.)
  print(enabled_branches['FS_Protons'][:,:,0])
  # Use a function to compute an interesting kinematic quantity.
  proton_ke_sums = lehad.proton_ke_sum(enabled_branches)
  print('Proton kinetic energies:')
  print(proton_ke_sums)
  
  # Once we've read data from disk into python columnar datasets and applied selections and whatnot,
  #   we can pass responsibility to matplotlib to make plots.
  fix, ax = plt.subplots()
  _, bins, patches = plt.hist(proton_ke_sums,np.arange(0,3.0,0.2),weights=weights/0.2)
  ax.set_xlabel("Sum of proton KE (GeV)")
  ax.set_ylabel("Differential cross section (cm^2/GeV)")
  plt.show()
  plt.close()
  
  print('All done!')